In [1]:
!pip install wandb -q
import wandb
wandb.login() # This will prompt you to enter an API key from your free W&B account

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhruvpatel654321 (dhruvpatel654321-university-of-huddersfield) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from typing import List, Tuple, Dict, Any, Optional
import wandb

# ==========================================
# 1. Reproducibility
# ==========================================
def set_seeds(seed: int = 42) -> None:
    """
    Ensures total reproducibility across all computational libraries.
    This guarantees that every run yields the exact same results.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 2. Data Handling & PyTorch Datasets
# ==========================================
class FoodFactsDataset(Dataset):
    """PyTorch Dataset for text classification."""

    def __init__(self, texts: List[str], labels: List[int], tokenizer: DistilBertTokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class DataHandler:
    """Manages the raw data and prepares it for specific continual learning stages."""

    def __init__(self, train_path: str, val_path: str):
        print("Loading and optimising dataset size...")
        df_train = pd.read_csv(train_path).dropna(subset=['text', 'label'])
        df_val = pd.read_csv(val_path).dropna(subset=['text', 'label'])

        # Shuffle and sample to ensure balanced, fast training
        self.df_train = df_train.sample(frac=1.0, random_state=42).groupby('label').head(1500).reset_index(drop=True)
        self.df_val = df_val.sample(frac=1.0, random_state=42).groupby('label').head(200).reset_index(drop=True)

    def get_stage_data(self, start_class: int, end_class: int) -> Tuple[List[str], List[int], List[str], List[int]]:
        """Extracts text and labels for a specific stage, removing Pandas from downstream tasks."""
        stage_train = self.df_train[(self.df_train['label'] >= start_class) & (self.df_train['label'] < end_class)]
        stage_val = self.df_val[(self.df_val['label'] >= start_class) & (self.df_val['label'] < end_class)]

        return (
            stage_train['text'].tolist(), stage_train['label'].tolist(),
            stage_val['text'].tolist(), stage_val['label'].tolist()
        )

    def get_replay_samples(self, start_class: int, end_class: int, samples_per_class: int) -> Tuple[List[str], List[int]]:
        """Selects a subset of current stage data to save for future replay."""
        stage_df = self.df_train[(self.df_train['label'] >= start_class) & (self.df_train['label'] < end_class)]
        memory_df = stage_df.sample(frac=1.0, random_state=42).groupby('label').head(samples_per_class)
        return memory_df['text'].tolist(), memory_df['label'].tolist()

# ==========================================
# 3. The Replay Buffer (PyTorch Native)
# ==========================================
class ReplayBuffer:
    """
    Maintains memory of previous tasks natively in Python lists to prevent
    expensive Pandas concatenations during the training loop.
    """
    def __init__(self):
        self.memory_texts: List[str] = []
        self.memory_labels: List[int] = []

    def update_memory(self, texts: List[str], labels: List[int]) -> None:
        """Appends new examples to the buffer."""
        self.memory_texts.extend(texts)
        self.memory_labels.extend(labels)

    def get_memory_dataset(self, tokenizer: DistilBertTokenizer) -> Optional[FoodFactsDataset]:
        """Returns the current buffer as a PyTorch Dataset, or None if empty."""
        if not self.memory_texts:
            return None
        return FoodFactsDataset(self.memory_texts, self.memory_labels, tokenizer)

# ==========================================
# 4. The Continual Learner Core
# ==========================================
class ContinualLearner:
    """Orchestrates the training, LoRA integration, evaluation, and experiment tracking."""

    def __init__(self, num_stages: int = 10, total_classes: int = 100, memory_size_per_class: int = 50):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.num_stages = num_stages
        self.total_classes = total_classes
        self.classes_per_stage = total_classes // num_stages
        self.memory_size_per_class = memory_size_per_class

        # Initialise Model & Tokenizer
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        base_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=total_classes)

        # PEFT / LoRA Configuration
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["q_lin", "k_lin", "v_lin"]
        )
        self.model = get_peft_model(base_model, lora_config).to(self.device)

        self.val_loaders_seen: List[DataLoader] = []
        self.accuracy_matrix: List[List[float]] = []
        self.memory_buffer = ReplayBuffer()

    def calculate_bwt(self) -> float:
        """Calculates Backward Transfer (BWT) using the accuracy matrix."""
        num_tasks = len(self.accuracy_matrix)
        if num_tasks < 2: return 0.0
        return sum((self.accuracy_matrix[-1][i] - self.accuracy_matrix[i][i]) for i in range(num_tasks - 1)) / (num_tasks - 1)

    def evaluate(self, current_stage: int) -> List[float]:
        """Evaluates the model across all tasks seen so far."""
        self.model.eval()
        accuracies = []
        active_classes = (current_stage + 1) * self.classes_per_stage

        with torch.no_grad():
            for task_id, val_loader in enumerate(self.val_loaders_seen):
                correct, total = 0, 0
                for batch in val_loader:
                    input_ids, attention_mask, labels = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device), batch['labels'].to(self.device)

                    logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
                    logits[:, active_classes:] = -float('inf') # Mask unseen classes

                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == labels).sum().item()
                    total += labels.size(0)

                task_acc = correct / total if total > 0 else 0.0
                accuracies.append(task_acc)
                print(f"    -> Accuracy on Task {task_id}: {task_acc * 100:.2f}%")

                # Log to Weights & Biases
                wandb.log({f"eval/Task_{task_id}_Accuracy": task_acc * 100, "stage": current_stage})

        return accuracies

    def train(self, data_handler: DataHandler, epochs_per_stage: int = 3) -> Tuple[nn.Module, List[List[float]]]:
        """Main training loop orchestrating data mixing and parameter updates."""

        # Initialise MLOps Tracking
        wandb.init(project="Roseway_KTP_Continual_Learning", config={
            "epochs": epochs_per_stage,
            "memory_size": self.memory_size_per_class,
            "model": "DistilBERT+LoRA"
        })

        for stage in range(self.num_stages):
            start_class = stage * self.classes_per_stage
            end_class = (stage + 1) * self.classes_per_stage
            print(f"\n{'='*40}\nSTAGE {stage}: Classes {start_class} to {end_class - 1}\n{'='*40}")

            # 1. Fetch Data
            train_texts, train_labels, val_texts, val_labels = data_handler.get_stage_data(start_class, end_class)
            current_task_dataset = FoodFactsDataset(train_texts, train_labels, self.tokenizer)

            # 2. Mix with Replay Buffer natively in PyTorch
            memory_dataset = self.memory_buffer.get_memory_dataset(self.tokenizer)
            if memory_dataset is not None:
                print(f"Mixing replay samples from previous stages...")
                combined_dataset = ConcatDataset([current_task_dataset, memory_dataset])
            else:
                combined_dataset = current_task_dataset

            train_loader = DataLoader(combined_dataset, batch_size=32, shuffle=True)

            val_dataset = FoodFactsDataset(val_texts, val_labels, self.tokenizer)
            self.val_loaders_seen.append(DataLoader(val_dataset, batch_size=32, shuffle=False))

            # 3. Model Training
            optimizer = AdamW(self.model.parameters(), lr=1e-4)
            loss_fn = nn.CrossEntropyLoss()

            self.model.train()
            for epoch in range(epochs_per_stage):
                total_loss = 0
                for batch in train_loader:
                    optimizer.zero_grad()
                    input_ids, attention_mask, labels = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device), batch['labels'].to(self.device)

                    logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
                    masked_logits = logits.clone()
                    masked_logits[:, end_class:] = -float('inf')

                    loss = loss_fn(masked_logits, labels)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()

                avg_loss = total_loss / len(train_loader)
                print(f"  Epoch {epoch + 1}/{epochs_per_stage} | Training Loss: {avg_loss:.4f}")
                wandb.log({"train/loss": avg_loss, "epoch_absolute": stage * epochs_per_stage + epoch})

            # 4. Update Memory Buffer & Evaluate
            mem_texts, mem_labels = data_handler.get_replay_samples(start_class, end_class, self.memory_size_per_class)
            self.memory_buffer.update_memory(mem_texts, mem_labels)

            print(f"\nEvaluating Stage {stage}...")
            stage_accuracies = self.evaluate(stage)
            self.accuracy_matrix.append(stage_accuracies)
            wandb.log({"eval/avg_accuracy_seen_tasks": np.mean(stage_accuracies) * 100, "stage": stage})

        bwt = self.calculate_bwt()
        print(f"\nFinal Backward Transfer (BWT): {bwt * 100:.2f}%")
        wandb.log({"final/BWT": bwt * 100})
        wandb.finish() # Closes the W&B run

        return self.model, self.accuracy_matrix

# ==========================================
# 5. Execution Pipeline
# ==========================================
if __name__ == "__main__":
    # 1. Enforce strict reproducibility
    set_seeds(42)

    # 2. Initialise the pipeline components
    train_csv_path = '/content/clean_train.csv' # UPDATE PATH
    val_csv_path = '/content/clean_val.csv'     # UPDATE PATH

    data_handler = DataHandler(train_csv_path, val_csv_path)
    learner = ContinualLearner(num_stages=10, total_classes=100, memory_size_per_class=50)

    # 3. Execute training
    final_model, final_matrix = learner.train(data_handler, epochs_per_stage=3)

    # 4. Display Results
    matrix_df = pd.DataFrame(final_matrix)
    matrix_df.index = [f"Trained on Stage {i}" for i in range(10)]
    matrix_df.columns = [f"Eval on Stage {i}" for i in range(10)]
    print("\n=== FINAL ACCURACY MATRIX ===")
    from IPython.display import display
    display(matrix_df.round(4) * 100)

Loading and optimising dataset size...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dhruvpatel654321 (dhruvpatel654321-university-of-huddersfield) to https://api.wandb.ai. Use `wandb login --relogin` t


STAGE 0: Classes 0 to 9
  Epoch 1/3 | Training Loss: 1.7442
  Epoch 2/3 | Training Loss: 1.2953
  Epoch 3/3 | Training Loss: 1.1452

Evaluating Stage 0...
    -> Accuracy on Task 0: 61.95%

STAGE 1: Classes 10 to 19
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.1987
  Epoch 2/3 | Training Loss: 0.7421
  Epoch 3/3 | Training Loss: 0.6495

Evaluating Stage 1...
    -> Accuracy on Task 0: 9.80%
    -> Accuracy on Task 1: 81.80%

STAGE 2: Classes 20 to 29
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.1712
  Epoch 2/3 | Training Loss: 0.7281
  Epoch 3/3 | Training Loss: 0.6230

Evaluating Stage 2...
    -> Accuracy on Task 0: 17.65%
    -> Accuracy on Task 1: 41.15%
    -> Accuracy on Task 2: 86.65%

STAGE 3: Classes 30 to 39
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.2838
  Epoch 2/3 | Training Loss: 0.8130
  Epoch 3/3 | Training Loss: 0.7165

Evaluating Stage 3...
    -> Accuracy on Task 0: 18

epoch_absolute,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/Task_0_Accuracy,█▁▂▂▂▃▄▄▄▃
eval/Task_1_Accuracy,█▂▁▃▄▃▄▄▄
eval/Task_2_Accuracy,█▁▁▄▃▃▃▃
eval/Task_3_Accuracy,█▁▃▃▃▃▃
eval/Task_4_Accuracy,█▁▁▂▂▁
eval/Task_5_Accuracy,█▄▁▄▅
eval/Task_6_Accuracy,█▂▂▁
eval/Task_7_Accuracy,█▁▇
eval/Task_8_Accuracy,▁█
+5,...



=== FINAL ACCURACY MATRIX ===


,Eval on Stage 0,Eval on Stage 1,Eval on Stage 2,Eval on Stage 3,Eval on Stage 4,Eval on Stage 5,Eval on Stage 6,Eval on Stage 7,Eval on Stage 8,Eval on Stage 9
Trained on Stage 0,61.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Trained on Stage 1,9.80,81.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Trained on Stage 2,17.65,41.15,86.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Trained on Stage 3,18.70,32.50,48.45,86.85,NaN,NaN,NaN,NaN,NaN,NaN
Trained on Stage 4,20.00,47.70,50.05,57.00,81.63,NaN,NaN,NaN,NaN,NaN
Trained on Stage 5,28.00,51.40,64.40,63.65,56.97,70.92,NaN,NaN,NaN,NaN
Trained on Stage 6,29.45,45.80,60.10,65.62,57.17,61.14,62.8,NaN,NaN,NaN
Trained on Stage 7,30.20,50.60,60.55,66.58,58.84,54.35,50.0,51.08,NaN,NaN
Trained on Stage 8,28.55,51.55,57.70,65.30,60.02,61.14,50.8,44.09,60.65,NaN
Trained on Stage 9,27.00,50.55,59.65,64.08,56.19,62.77,48.8,50.54,65.16,56.72


In [5]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from typing import List, Tuple, Dict, Optional
import wandb

# ==========================================
# 1. Reproducibility
# ==========================================
def set_seeds(seed: int = 42) -> None:
    """
    Ensures total reproducibility across all computational libraries.
    This guarantees that every run yields the exact same results.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 2. Data Handling & PyTorch Datasets
# ==========================================
class FoodFactsDataset(Dataset):
    """PyTorch Dataset for text classification."""

    def __init__(self, texts: List[str], labels: List[int], tokenizer: DistilBertTokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class DataHandler:
    """Manages the raw data and prepares it for specific continual learning stages."""

    def __init__(self, train_path: str, val_path: str):
        print("Loading and optimising dataset size...")
        df_train = pd.read_csv(train_path).dropna(subset=['text', 'label'])
        df_val = pd.read_csv(val_path).dropna(subset=['text', 'label'])

        # Shuffle and sample to ensure balanced, fast training
        self.df_train = df_train.sample(frac=1.0, random_state=42).groupby('label').head(1500).reset_index(drop=True)
        self.df_val = df_val.sample(frac=1.0, random_state=42).groupby('label').head(200).reset_index(drop=True)

    def get_stage_data(self, start_class: int, end_class: int) -> Tuple[List[str], List[int], List[str], List[int]]:
        """Extracts text and labels for a specific stage, removing Pandas from downstream tasks."""
        stage_train = self.df_train[(self.df_train['label'] >= start_class) & (self.df_train['label'] < end_class)]
        stage_val = self.df_val[(self.df_val['label'] >= start_class) & (self.df_val['label'] < end_class)]

        return (
            stage_train['text'].tolist(), stage_train['label'].tolist(),
            stage_val['text'].tolist(), stage_val['label'].tolist()
        )

    def get_replay_samples(self, start_class: int, end_class: int, samples_per_class: int) -> Tuple[List[str], List[int]]:
        """Selects a subset of current stage data to save for future replay."""
        stage_df = self.df_train[(self.df_train['label'] >= start_class) & (self.df_train['label'] < end_class)]
        memory_df = stage_df.sample(frac=1.0, random_state=42).groupby('label').head(samples_per_class)
        return memory_df['text'].tolist(), memory_df['label'].tolist()

# ==========================================
# 3. The Replay Buffer (PyTorch Native)
# ==========================================
class ReplayBuffer:
    """
    Maintains memory of previous tasks natively in Python lists to prevent
    expensive Pandas concatenations during the training loop.
    """
    def __init__(self):
        self.memory_texts: List[str] = []
        self.memory_labels: List[int] = []

    def update_memory(self, texts: List[str], labels: List[int]) -> None:
        """Appends new examples to the buffer."""
        self.memory_texts.extend(texts)
        self.memory_labels.extend(labels)

    def get_memory_dataset(self, tokenizer: DistilBertTokenizer) -> Optional[FoodFactsDataset]:
        """Returns the current buffer as a PyTorch Dataset, or None if empty."""
        if not self.memory_texts:
            return None
        return FoodFactsDataset(self.memory_texts, self.memory_labels, tokenizer)

# ==========================================
# 4. The Continual Learner Core
# ==========================================
class ContinualLearner:
    """Orchestrates the training, LoRA integration, evaluation, and experiment tracking."""

    def __init__(self, num_stages: int = 10, total_classes: int = 100, memory_size_per_class: int = 50):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.num_stages = num_stages
        self.total_classes = total_classes
        self.classes_per_stage = total_classes // num_stages
        self.memory_size_per_class = memory_size_per_class

        # Initialise a fresh Model & Tokenizer for every instance to prevent weight leakage
        self.tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        base_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=total_classes)

        # PEFT / LoRA Configuration
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["q_lin", "k_lin", "v_lin"]
        )
        self.model = get_peft_model(base_model, lora_config).to(self.device)

        self.val_loaders_seen: List[DataLoader] = []
        self.accuracy_matrix: List[List[float]] = []
        self.memory_buffer = ReplayBuffer()

    def calculate_bwt(self) -> float:
        """Calculates Backward Transfer (BWT) using the accuracy matrix."""
        num_tasks = len(self.accuracy_matrix)
        if num_tasks < 2: return 0.0
        return sum((self.accuracy_matrix[-1][i] - self.accuracy_matrix[i][i]) for i in range(num_tasks - 1)) / (num_tasks - 1)

    def evaluate(self, current_stage: int) -> List[float]:
        """Evaluates the model across all tasks seen so far."""
        self.model.eval()
        accuracies = []
        active_classes = (current_stage + 1) * self.classes_per_stage

        with torch.no_grad():
            for task_id, val_loader in enumerate(self.val_loaders_seen):
                correct, total = 0, 0
                for batch in val_loader:
                    input_ids, attention_mask, labels = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device), batch['labels'].to(self.device)

                    logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
                    logits[:, active_classes:] = -float('inf') # Mask unseen classes

                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == labels).sum().item()
                    total += labels.size(0)

                task_acc = correct / total if total > 0 else 0.0
                accuracies.append(task_acc)
                print(f"    -> Accuracy on Task {task_id}: {task_acc * 100:.2f}%")

                # Log to Weights & Biases
                wandb.log({f"eval/Task_{task_id}_Accuracy": task_acc * 100, "stage": current_stage})

        return accuracies

    def train(self, data_handler: DataHandler, use_replay: bool = False, epochs_per_stage: int = 3) -> Tuple[nn.Module, List[List[float]]]:
        """Main training loop orchestrating data mixing and parameter updates."""

        run_name = "Experience_Replay" if use_replay else "Naive_Baseline"
        wandb.init(project="Roseway_KTP_Continual_Learning", name=run_name, config={
            "epochs": epochs_per_stage,
            "memory_size": self.memory_size_per_class if use_replay else 0,
            "model": "DistilBERT+LoRA",
            "strategy": run_name
        })

        for stage in range(self.num_stages):
            start_class = stage * self.classes_per_stage
            end_class = (stage + 1) * self.classes_per_stage
            print(f"\n{'='*40}\nSTAGE {stage}: Classes {start_class} to {end_class - 1}\n{'='*40}")

            # 1. Fetch Data
            train_texts, train_labels, val_texts, val_labels = data_handler.get_stage_data(start_class, end_class)
            current_task_dataset = FoodFactsDataset(train_texts, train_labels, self.tokenizer)

            # 2. Mix with Replay Buffer (ONLY if use_replay is True)
            if use_replay:
                memory_dataset = self.memory_buffer.get_memory_dataset(self.tokenizer)
                if memory_dataset is not None:
                    print(f"Mixing replay samples from previous stages...")
                    combined_dataset = ConcatDataset([current_task_dataset, memory_dataset])
                else:
                    combined_dataset = current_task_dataset
            else:
                combined_dataset = current_task_dataset

            train_loader = DataLoader(combined_dataset, batch_size=32, shuffle=True)

            val_dataset = FoodFactsDataset(val_texts, val_labels, self.tokenizer)
            self.val_loaders_seen.append(DataLoader(val_dataset, batch_size=32, shuffle=False))

            # 3. Model Training
            optimizer = AdamW(self.model.parameters(), lr=1e-4)
            loss_fn = nn.CrossEntropyLoss()

            self.model.train()
            for epoch in range(epochs_per_stage):
                total_loss = 0
                for batch in train_loader:
                    optimizer.zero_grad()
                    input_ids, attention_mask, labels = batch['input_ids'].to(self.device), batch['attention_mask'].to(self.device), batch['labels'].to(self.device)

                    logits = self.model(input_ids=input_ids, attention_mask=attention_mask).logits
                    masked_logits = logits.clone()
                    masked_logits[:, end_class:] = -float('inf')

                    loss = loss_fn(masked_logits, labels)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()

                avg_loss = total_loss / len(train_loader)
                print(f"  Epoch {epoch + 1}/{epochs_per_stage} | Training Loss: {avg_loss:.4f}")
                wandb.log({"train/loss": avg_loss, "epoch_absolute": stage * epochs_per_stage + epoch})

            # 4. Update Memory Buffer (ONLY if use_replay is True)
            if use_replay:
                mem_texts, mem_labels = data_handler.get_replay_samples(start_class, end_class, self.memory_size_per_class)
                self.memory_buffer.update_memory(mem_texts, mem_labels)

            print(f"\nEvaluating Stage {stage}...")
            stage_accuracies = self.evaluate(stage)
            self.accuracy_matrix.append(stage_accuracies)
            wandb.log({"eval/avg_accuracy_seen_tasks": np.mean(stage_accuracies) * 100, "stage": stage})

        bwt = self.calculate_bwt()
        print(f"\nFinal Backward Transfer (BWT): {bwt * 100:.2f}%")
        wandb.log({"final/BWT": bwt * 100})
        wandb.finish() # Closes the W&B run

        return self.model, self.accuracy_matrix

# ==========================================
# 5. Execution Pipeline
# ==========================================
if __name__ == "__main__":
    from IPython.display import display

    # 1. Enforce strict reproducibility
    set_seeds(42)

    # 2. Initialise Data
    train_csv_path = '/content/clean_train.csv'
    val_csv_path = '/content/clean_val.csv'

    if not os.path.exists(train_csv_path) or not os.path.exists(val_csv_path):
        print("ERROR: clean_train.csv and clean_val.csv must be in the same folder as this script.")
    else:
        data_handler = DataHandler(train_csv_path, val_csv_path)

        # ----------------------------------------------------
        # EXPERIMENT 1: NAÏVE BASELINE
        # ----------------------------------------------------
        print("\n\n" + "#"*50)
        print("COMMENCING EXPERIMENT 1: NAÏVE BASELINE")
        print("#"*50)
        baseline_learner = ContinualLearner(num_stages=10, total_classes=100)
        _, baseline_matrix = baseline_learner.train(data_handler, use_replay=False, epochs_per_stage=3)

        df_baseline = pd.DataFrame(baseline_matrix)
        df_baseline.index = [f"Trained on Stage {i}" for i in range(10)]
        df_baseline.columns = [f"Eval on Stage {i}" for i in range(10)]
        print("\n=== NAÏVE BASELINE ACCURACY MATRIX ===")
        print(df_baseline.round(4) * 100)

        # ----------------------------------------------------
        # EXPERIMENT 2: EXPERIENCE REPLAY
        # ----------------------------------------------------
        print("\n\n" + "#"*50)
        print("COMMENCING EXPERIMENT 2: EXPERIENCE REPLAY")
        print("#"*50)
        # Note: A completely fresh ContinualLearner is initialised to ensure zero weight leakage
        replay_learner = ContinualLearner(num_stages=10, total_classes=100, memory_size_per_class=50)
        _, replay_matrix = replay_learner.train(data_handler, use_replay=True, epochs_per_stage=3)

        df_replay = pd.DataFrame(replay_matrix)
        df_replay.index = [f"Trained on Stage {i}" for i in range(10)]
        df_replay.columns = [f"Eval on Stage {i}" for i in range(10)]
        print("\n=== EXPERIENCE REPLAY ACCURACY MATRIX ===")
        print(df_replay.round(4) * 100)

Loading and optimising dataset size...


##################################################
COMMENCING EXPERIMENT 1: NAÏVE BASELINE
##################################################


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



STAGE 0: Classes 0 to 9
  Epoch 1/3 | Training Loss: 1.7442
  Epoch 2/3 | Training Loss: 1.2953
  Epoch 3/3 | Training Loss: 1.1452

Evaluating Stage 0...
    -> Accuracy on Task 0: 61.95%

STAGE 1: Classes 10 to 19
  Epoch 1/3 | Training Loss: 1.0761
  Epoch 2/3 | Training Loss: 0.6101
  Epoch 3/3 | Training Loss: 0.5261

Evaluating Stage 1...
    -> Accuracy on Task 0: 0.00%
    -> Accuracy on Task 1: 82.60%

STAGE 2: Classes 20 to 29
  Epoch 1/3 | Training Loss: 1.0598
  Epoch 2/3 | Training Loss: 0.5373
  Epoch 3/3 | Training Loss: 0.4511

Evaluating Stage 2...
    -> Accuracy on Task 0: 0.00%
    -> Accuracy on Task 1: 0.45%
    -> Accuracy on Task 2: 87.10%

STAGE 3: Classes 30 to 39
  Epoch 1/3 | Training Loss: 1.1522
  Epoch 2/3 | Training Loss: 0.5345
  Epoch 3/3 | Training Loss: 0.4367

Evaluating Stage 3...
    -> Accuracy on Task 0: 0.00%
    -> Accuracy on Task 1: 0.00%
    -> Accuracy on Task 2: 0.20%
    -> Accuracy on Task 3: 88.66%

STAGE 4: Classes 40 to 49
  Epoch 1

epoch_absolute,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/Task_0_Accuracy,█▁▁▁▁▁▁▁▁▁
eval/Task_1_Accuracy,█▁▁▁▁▁▁▁▁
eval/Task_2_Accuracy,█▁▁▁▁▁▁▁
eval/Task_3_Accuracy,█▁▁▁▁▁▁
eval/Task_4_Accuracy,█▂▁▁▁▁
eval/Task_5_Accuracy,█▂▁▁▁
eval/Task_6_Accuracy,█▂▁▁
eval/Task_7_Accuracy,█▁▁
eval/Task_8_Accuracy,█▁
+5,...


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



STAGE 0: Classes 0 to 9
  Epoch 1/3 | Training Loss: 1.7380
  Epoch 2/3 | Training Loss: 1.2825
  Epoch 3/3 | Training Loss: 1.1303

Evaluating Stage 0...
    -> Accuracy on Task 0: 62.80%

STAGE 1: Classes 10 to 19
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.2025
  Epoch 2/3 | Training Loss: 0.7329
  Epoch 3/3 | Training Loss: 0.6426

Evaluating Stage 1...
    -> Accuracy on Task 0: 10.75%
    -> Accuracy on Task 1: 82.30%

STAGE 2: Classes 20 to 29
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.1626
  Epoch 2/3 | Training Loss: 0.7209
  Epoch 3/3 | Training Loss: 0.6224

Evaluating Stage 2...
    -> Accuracy on Task 0: 20.90%
    -> Accuracy on Task 1: 42.65%
    -> Accuracy on Task 2: 86.50%

STAGE 3: Classes 30 to 39
Mixing replay samples from previous stages...
  Epoch 1/3 | Training Loss: 1.2895
  Epoch 2/3 | Training Loss: 0.8199
  Epoch 3/3 | Training Loss: 0.7181

Evaluating Stage 3...
    -> Accuracy on Task 0: 1

epoch_absolute,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
eval/Task_0_Accuracy,█▁▂▂▃▄▃▃▃▃
eval/Task_1_Accuracy,█▂▁▂▃▃▃▃▃
eval/Task_2_Accuracy,█▁▂▄▄▄▃▃
eval/Task_3_Accuracy,█▁▃▃▃▃▃
eval/Task_4_Accuracy,█▂▂▂▃▁
eval/Task_5_Accuracy,█▂▁▄▃
eval/Task_6_Accuracy,█▁▃▂
eval/Task_7_Accuracy,█▁▇
eval/Task_8_Accuracy,█▁
+5,...



=== EXPERIENCE REPLAY ACCURACY MATRIX ===
                    Eval on Stage 0  Eval on Stage 1  Eval on Stage 2  \
Trained on Stage 0            62.80              NaN              NaN   
Trained on Stage 1            10.75            82.30              NaN   
Trained on Stage 2            20.90            42.65            86.50   
Trained on Stage 3            15.40            34.80            45.30   
Trained on Stage 4            23.60            43.80            48.95   
Trained on Stage 5            29.35            49.75            63.90   
Trained on Stage 6            28.45            46.05            62.85   
Trained on Stage 7            28.00            50.50            62.85   
Trained on Stage 8            27.35            45.55            57.65   
Trained on Stage 9            27.60            47.65            58.60   

                    Eval on Stage 3  Eval on Stage 4  Eval on Stage 5  \
Trained on Stage 0              NaN              NaN              NaN   
Trained